In [1]:
#!/usr/init/env python3
# -*- coding: utf-8 -*-

###: ITSxpress
### Paired-End  
### ITS Amplicon Sequencing


import sys
import subprocess
import time
import gzip
from pathlib import Path
import subprocess

# Capture Python version
python_version = sys.version.split()[0]

In [2]:
# # 1. Initializing environment and checking software versions-----------
# Capture Python version
python_version = sys.version.split()[0]

# Verify that ITSxpress is installed and accessible in the system PATH
try:
    version_check = subprocess.run(["itsxpress", "--version"], capture_output=True, text=True, check=True)
    itsxpress_version = version_check.stdout.strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    raise RuntimeError("ITSxpress is not installed or not available in the current environment PATH.")

# Define main project base path
base_path = Path("/home/tatiana/PROYECTOS/repository/sequences/")  ## define your path
 
# Define subdirectories safely relative to base_path
input_dir = base_path / "its_results/cutadapt"
output_dir = base_path / "its_results/itsxpress"
reports_dir = base_path / "reports"

# Automatically create output directories if they don't exist
output_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Search for all R1 FASTQ files matching the pattern
fastq_files_R1 = sorted(list(input_dir.glob("*_R1_001.fastq.gz")))

# Validate that files were actually found before proceeding
if not fastq_files_R1:
    raise FileNotFoundError(f"No FASTQ files found in '{input_dir}'. Please verify your folder path.")

# # Show the first 3 files found in the input folder to visually confirm content
[f.name for f in fastq_files_R1[:3]]

['its-LC1_S5_L001_R1_001.fastq.gz',
 'its-LC2_S7_L001_R1_001.fastq.gz',
 'its-LC3_S9_L001_R1_001.fastq.gz']

In [3]:
# # 2. Pilot test on the first sample-----------
# Select the first sample from the R1 list for testing
r1_test = fastq_files_R1[0]

# Define the corresponding R2 file by replacing the name pattern
r2_test = Path(str(r1_test).replace("_R1_001.fastq.gz", "_R2_001.fastq.gz"))

# Set up output and log paths for the pilot test
out_r1_test = output_dir / r1_test.name
out_r2_test = output_dir / r2_test.name
log_test = output_dir / f"{r1_test.stem}_test.log"

# Build the terminal command list for paired-end ITS1 Fungi processing
cmd_test = [
    "itsxpress",
    "--fastq", str(r1_test),
    "--fastq2", str(r2_test),
    "--region", "ITS1",
    "--taxa", "Fungi",
    "--outfile", str(out_r1_test),
    "--outfile2", str(out_r2_test),
    "--threads", "4",
    "--log", str(log_test)
]

# Execute the pilot command securely
resultado = subprocess.run(cmd_test, capture_output=True, text=True)

# Raise an exception with error details if the pilot test fails
if resultado.returncode != 0:
    raise RuntimeError(f"Pilot test failed for {r1_test.name}:\n{resultado.stderr}")

# Return the name of the successfully tested sample as visual confirmation
r1_test.name

'its-LC1_S5_L001_R1_001.fastq.gz'

In [4]:
# # 3. Batch processing all samples-----------
# Iterate through all R1 FASTQ files found in the input directory
for r1_file in fastq_files_R1:
    
    # Define the corresponding R2 file path dynamically
    r2_file = Path(str(r1_file).replace("_R1_001.fastq.gz", "_R2_001.fastq.gz"))
    
    # Define output and log file paths for the current sample
    out_r1 = output_dir / r1_file.name
    out_r2 = output_dir / r2_file.name
    log_file = output_dir / f"{r1_file.stem}_itsxpress.log"
    
    # Construct the terminal command list for ITSxpress batch processing
    cmd_loop = [
        "itsxpress",
        "--fastq", str(r1_file),
        "--fastq2", str(r2_file),
        "--region", "ITS1",
        "--taxa", "Fungi",
        "--outfile", str(out_r1),
        "--outfile2", str(out_r2),
        "--threads", "4",
        "--log", str(log_file)
    ]
    
    # Execute the command for each sample
    resultado_loop = subprocess.run(cmd_loop, capture_output=True, text=True)
    
    # Raise an error immediately if any sample fails during processing
    if resultado_loop.returncode != 0:
        raise RuntimeError(f"Error processing sample {r1_file.name}:\n{resultado_loop.stderr}")

# List the processed output files as a clean visual confirmation when complete
[f.name for f in sorted(list(output_dir.glob("*_R1_001.fastq.gz")))]

['its-LC1_S5_L001_R1_001.fastq.gz',
 'its-LC2_S7_L001_R1_001.fastq.gz',
 'its-LC3_S9_L001_R1_001.fastq.gz',
 'its-LL1_S2_L001_R1_001.fastq.gz',
 'its-LL2_S3_L001_R1_001.fastq.gz',
 'its-LL3_S4_L001_R1_001.fastq.gz',
 'its-LRS1_S6_L001_R1_001.fastq.gz',
 'its-LRS2_S8_L001_R1_001.fastq.gz',
 'its-LRS3_S10_L001_R1_001.fastq.gz',
 'its-negC_S1_L001_R1_001.fastq.gz',
 'its-posC_S11_L001_R1_001.fastq.gz']

In [6]:
# 4. Summary table and CSV export using standard library-----------
import csv

# Collect summary data by checking processed file names and sizes
summary_data = []
for r1_out in sorted(list(output_dir.glob("*_R1_001.fastq.gz"))):
    sample_name = r1_out.name.split("_R1_001.fastq.gz")[0]
    file_size_mb = round(r1_out.stat().st_size / (1024 * 1024), 2)
    summary_data.append({
        "Sample": sample_name,
        "Processed_R1_Size_MB": file_size_mb
    })

# Define export path for the CSV report
csv_report_path = reports_dir / "itsxpress_processing_summary.csv"

# Write summary data to CSV using Python's built-in csv module
with open(csv_report_path, mode="w", newline="", encoding="utf-8") as csv_file:
    fieldnames = ["Sample", "Processed_R1_Size_MB"]
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    for row in summary_data:
        writer.writerow(row)

# Return the summary data list visually at the end of the cell
summary_data

[{'Sample': 'its-LC1_S5_L001', 'Processed_R1_Size_MB': 22.79},
 {'Sample': 'its-LC2_S7_L001', 'Processed_R1_Size_MB': 43.03},
 {'Sample': 'its-LC3_S9_L001', 'Processed_R1_Size_MB': 14.14},
 {'Sample': 'its-LL1_S2_L001', 'Processed_R1_Size_MB': 22.86},
 {'Sample': 'its-LL2_S3_L001', 'Processed_R1_Size_MB': 10.87},
 {'Sample': 'its-LL3_S4_L001', 'Processed_R1_Size_MB': 21.58},
 {'Sample': 'its-LRS1_S6_L001', 'Processed_R1_Size_MB': 3.51},
 {'Sample': 'its-LRS2_S8_L001', 'Processed_R1_Size_MB': 10.53},
 {'Sample': 'its-LRS3_S10_L001', 'Processed_R1_Size_MB': 59.21},
 {'Sample': 'its-negC_S1_L001', 'Processed_R1_Size_MB': 0.02},
 {'Sample': 'its-posC_S11_L001', 'Processed_R1_Size_MB': 10.85}]

In [7]:
# # 5. Check log files for zero-size samples-----------
log_files = sorted(list(output_dir.glob("*_itsxpress.log")))

# Read the last 10 lines of each log file to check for errors or zero matches
log_summaries = {}
for log in log_files:
    with open(log, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
        # Keep the last few lines where summary stats or errors usually appear
        log_summaries[log.name] = "".join(lines[-10:]) if lines else "Empty log"

# Return log summaries visually to inspect what happened
log_summaries

{'its-LC1_S5_L001_R1_001.fastq_itsxpress.log': '10-24 22:59 root         DEBUG    No ITS stop or start sites were identified for sequence M05074:547:000000000-LJJ8H:1:1101:6575:21695, skipping.\n10-24 22:59 root         DEBUG    No ITS stop or start sites were identified for sequence M05074:547:000000000-LJJ8H:1:1101:6575:21695, skipping.\n10-24 22:59 root         DEBUG    No ITS stop or start sites were identified for sequence M05074:547:000000000-LJJ8H:1:1101:6575:21695, skipping.\n10-24 22:59 root         DEBUG    No ITS stop or start sites were identified for sequence M05074:547:000000000-LJJ8H:1:1101:6575:21695, skipping.\n10-24 23:00 root         INFO     Counting reads after trimming.\n10-24 23:00 root         INFO     Total number of reads in file /home/tatiana/PROYECTOS/repository/sequences/its_results/cutadapt/its-LC1_S5_L001_R1_001.fastq.gz is 452417.\n10-24 23:00 root         INFO     Total number of reads in file /home/tatiana/PROYECTOS/repository/sequences/its_results/cut